# 01 — Terraform Foundation (v0.1)

Demonstra os módulos Terraform que formam o núcleo de IaC do projeto:
`networking` (VPC/subnets), `compute` (EC2/ASG/ALB), `database` (RDS MySQL),
`iam` (roles least-privilege + OIDC GitHub Actions), `security` (security
groups) e `monitoring` (CloudWatch).

**Nada neste notebook roda `terraform apply`.** Apenas `terraform fmt
-check`, `terraform init -backend=false` e `terraform validate` — comandos
somente de validação estática, que não criam nem modificam nenhum recurso
na AWS. `terraform init` baixa o *plugin* do provider AWS do registry
público (não usa credenciais AWS); se não houver conexão com a internet, o
notebook degrada graciosamente e explica o que não pôde ser validado.

In [1]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing PROJECT_LOG.md
    is found. Works whether the notebook is executed from notebooks/ (the
    normal case) or from the repo root."""
    p = start.resolve()
    for _ in range(8):
        if (p / "PROJECT_LOG.md").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not locate project root (PROJECT_LOG.md not found upward from %s)" % start)

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Caterpillar (Terminar)


In [2]:
import shutil
import subprocess

TERRAFORM_BIN = shutil.which("terraform")
if TERRAFORM_BIN:
    version_out = subprocess.run([TERRAFORM_BIN, "version"], capture_output=True, text=True, timeout=30)
    print(f"terraform encontrado em: {TERRAFORM_BIN}")
    print(version_out.stdout.strip())
else:
    print("terraform NÃO encontrado no PATH — as células seguintes vão pular a validação real "
          "e apenas descrever os módulos a partir dos arquivos .tf.")

terraform encontrado em: C:\Users\Yuri_\AppData\Local\Microsoft\WinGet\Packages\Hashicorp.Terraform_Microsoft.Winget.Source_8wekyb3d8bbwe\terraform.EXE
Terraform v1.15.8
on windows_amd64

Your version of Terraform is out of date! The latest version
is 1.15.9. You can update by downloading from https://developer.hashicorp.com/terraform/install


## `terraform fmt -check` (recursivo, todos os módulos)

In [3]:
TERRAFORM_DIR = PROJECT_ROOT / "terraform"

if TERRAFORM_BIN:
    fmt = subprocess.run(
        [TERRAFORM_BIN, "fmt", "-check", "-recursive", "-diff"],
        cwd=TERRAFORM_DIR, capture_output=True, text=True, timeout=60,
    )
    if fmt.returncode == 0:
        print("OK — todos os arquivos .tf já estão formatados (terraform fmt -check passou).")
    else:
        print("Arquivos fora do padrão de formatação (terraform fmt -diff):\n")
        print(fmt.stdout or fmt.stderr)
else:
    print("terraform indisponível — pulando fmt -check.")

OK — todos os arquivos .tf já estão formatados (terraform fmt -check passou).


## Módulos e variáveis declaradas (leitura estática dos arquivos .tf)

In [4]:
import re

MODULES_DIR = TERRAFORM_DIR / "modules"
var_pattern = re.compile(r'variable\s+"([a-zA-Z0-9_]+)"')

for module_dir in sorted(p for p in MODULES_DIR.iterdir() if p.is_dir()):
    vars_file = module_dir / "variables.tf"
    outputs_file = module_dir / "outputs.tf"
    n_vars = len(var_pattern.findall(vars_file.read_text(encoding="utf-8"))) if vars_file.exists() else 0
    n_outputs = len(re.findall(r'output\s+"([a-zA-Z0-9_]+)"', outputs_file.read_text(encoding="utf-8"))) if outputs_file.exists() else 0
    print(f"{module_dir.name:12s} -> {n_vars:2d} variable(s), {n_outputs:2d} output(s)")

compute      -> 22 variable(s),  8 output(s)
database     -> 17 variable(s),  9 output(s)
iam          -> 11 variable(s),  5 output(s)


monitoring   -> 20 variable(s),  6 output(s)
networking   ->  8 variable(s),  8 output(s)


security     -> 10 variable(s),  3 output(s)


## Papel de cada módulo

| Módulo | Recursos AWS principais | Responsabilidade |
|--------|--------------------------|-------------------|
| `networking` | VPC, subnets públicas/privadas, Internet Gateway, NAT Gateway (opcional), route tables | Isolamento de rede — base de tudo o mais |
| `security` | Security Groups (web/app/db) | Least-privilege: ALB só expõe 80/443, app só aceita tráfego do SG do ALB, DB só aceita tráfego do SG da app |
| `iam` | IAM Role/Instance Profile para EC2 (SSM, sem SSH aberto), OIDC provider + Role para GitHub Actions | Elimina chaves de acesso de longa duração; instâncias usam SSM Session Manager, CI usa OIDC |
| `compute` | Launch Template, Auto Scaling Group, Application Load Balancer + Target Group | Camada de aplicação (Linux ou Windows), escalável e balanceada |
| `database` | RDS MySQL (Multi-AZ opcional, senha gerenciada no Secrets Manager) | Persistência (execuções/incidentes/inventário) |
| `monitoring` | CloudWatch Alarms (CPU, 5xx, storage), SNS (opcional), Log Groups | Observabilidade — alimenta o dashboard e alerta antes de virar incidente |

## `terraform init -backend=false` + `terraform validate` (root sintético local)

In [5]:
"""
Os módulos em terraform/modules/ não são "root modules" completos (não
declaram provider próprio) -- em produção eles são chamados a partir de
terraform/environments/<env>/main.tf. Como environments/ ainda está sendo
completado por outra trilha em paralelo, montamos aqui um root sintético
TEMPORÁRIO (em uma pasta temp, nunca dentro do repo) que referencia os 6
módulos reais via caminho relativo, com valores plausíveis de variáveis --
o suficiente para validar que os módulos são estruturalmente corretos, sem
nunca fazer apply.
"""
import shutil as _shutil
import tempfile
import textwrap

SYNTH_MAIN_TF = textwrap.dedent('''
    terraform {
      required_version = ">= 1.5"
      required_providers {
        aws = {
          source  = "hashicorp/aws"
          version = "~> 5.0"
        }
      }
    }

    provider "aws" {
      region                      = "sa-east-1"
      access_key                  = "test"
      secret_key                  = "test"
      skip_credentials_validation = true
      skip_requesting_account_id  = true
      skip_metadata_api_check     = true
    }

    module "networking" {
      source               = "./modules/networking"
      name_prefix          = "notebook-validate"
      vpc_cidr             = "10.0.0.0/16"
      azs                  = ["sa-east-1a", "sa-east-1b"]
      public_subnet_cidrs  = ["10.0.0.0/24", "10.0.1.0/24"]
      private_subnet_cidrs = ["10.0.10.0/24", "10.0.11.0/24"]
    }

    module "security" {
      source      = "./modules/security"
      name_prefix = "notebook-validate"
      vpc_id      = module.networking.vpc_id
    }

    module "iam" {
      source       = "./modules/iam"
      name_prefix  = "notebook-validate"
      github_org   = "yuri-dubbern"
      github_repo  = "enterprise-cloud-automation"
    }

    module "compute" {
      source                 = "./modules/compute"
      name_prefix            = "notebook-validate"
      vpc_id                 = module.networking.vpc_id
      public_subnet_ids      = module.networking.public_subnet_ids
      instance_subnet_ids    = module.networking.public_subnet_ids
      web_security_group_id  = module.security.web_security_group_id
      app_security_group_id  = module.security.app_security_group_id
      iam_instance_profile_name = module.iam.ec2_instance_profile_name
    }

    module "database" {
      source                 = "./modules/database"
      name_prefix             = "notebook-validate"
      subnet_ids              = module.networking.private_subnet_ids
      vpc_security_group_ids  = [module.security.db_security_group_id]
    }

    module "monitoring" {
      source                  = "./modules/monitoring"
      name_prefix              = "notebook-validate"
      enable_ec2_alarms        = true
      autoscaling_group_name   = module.compute.autoscaling_group_name
      enable_alb_alarms        = true
      alb_arn_suffix           = module.compute.alb_arn_suffix
      target_group_arn_suffix  = module.compute.target_group_arn_suffix
      enable_rds_alarms        = true
      db_instance_id           = module.database.db_instance_id
    }
''').strip()

terraform_ok = False
validate_output = ""

if TERRAFORM_BIN:
    tmp_dir = Path(tempfile.mkdtemp(prefix="tf_validate_"))
    try:
        (tmp_dir / "main.tf").write_text(SYNTH_MAIN_TF, encoding="utf-8")
        # Symlink/copy the real modules directory in read-only fashion (copy is
        # simplest and safest cross-platform on Windows).
        _shutil.copytree(MODULES_DIR, tmp_dir / "modules")

        init = subprocess.run(
            [TERRAFORM_BIN, "init", "-backend=false", "-input=false"],
            cwd=tmp_dir, capture_output=True, text=True, timeout=180,
        )
        if init.returncode != 0:
            print("terraform init falhou (provável falta de conexão com o registry). Saída:\n")
            print(init.stdout[-2000:] + "\n" + init.stderr[-2000:])
        else:
            validate = subprocess.run(
                [TERRAFORM_BIN, "validate"],
                cwd=tmp_dir, capture_output=True, text=True, timeout=120,
            )
            validate_output = validate.stdout + validate.stderr
            terraform_ok = validate.returncode == 0
            print(validate_output.strip())
    except Exception as exc:  # pragma: no cover - defensive, notebook must never crash
        print(f"Não foi possível validar via terraform (ambiente sem rede/terraform?): {exc}")
    finally:
        _shutil.rmtree(tmp_dir, ignore_errors=True)
else:
    print("terraform indisponível — pulando init/validate. Módulos foram apenas descritos estaticamente acima.")

print()
print("Resultado:", "PASSOU" if terraform_ok else "não validado (ver mensagens acima)")

Success! The configuration is valid.




Resultado: PASSOU


## Resumo

- `terraform fmt -check -recursive`: valida estilo/formatação em todo `terraform/`.
- `terraform validate` (root sintético): confirma que os 6 módulos se
  conectam entre si sem erros de tipo/referência (VPC -> SGs -> IAM ->
  compute -> database -> monitoring), sem nunca criar um recurso real.
- Nenhuma credencial AWS real foi usada (`skip_credentials_validation =
  true`) e nenhum `apply`/`plan` contra a AWS de verdade foi executado.
- O `apply` real (ambiente `dev`) é uma decisão manual e documentada em
  [`docs/setup/aws-setup.md`](../docs/setup/aws-setup.md).